# 03 - LightGCN Baseline

Train and evaluate a compact LightGCN collaborative-filtering baseline on the
MovieLens-1M cold-start protocol produced by notebook 02. This notebook uses
interactions only: no title, genre, user demographic, or item-count features.

## Notebook Linkage and Work Plan

**Input from notebook 02:** the `ml1m-coldstart-v1` protocol manifest plus
verified `tuning_train`, `final_train`, `validation_tasks`, `evaluation_tasks`,
`users`, and `items` CSV artifacts. Those files are the dataset interface for
this baseline, so no separate dataset notebook is needed here.

**What this notebook does:**
1. Verify and load the protocol handoff.
2. Define a minimal LightGCN model, BPR training loop, phase graph builder, and
   row-level CTR metrics.
3. For each target seed, tune only on `tuning_train` plus validation support
   rows, selecting one configuration and one F1 threshold per phase from
   validation query labels.
4. Refit that seed's selected configuration on `final_train`, evaluate
   Cold/Warm A/B/C on new items, and apply the frozen validation thresholds.
5. Publish and verify one immutable LightGCN generation per target seed for
   notebooks 06-08.

## Verified Protocol Load

Locate notebook 02 locally or from a mounted Kaggle input, verify the pointer
manifest, immutable generation manifest, artifact hashes, row counts, columns,
and declared dtypes, then load the exact tables. This preserves the leakage
boundary: LightGCN sees only protocol-approved rows and indexed identifiers.

In [1]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import os
import platform
import random
import shutil
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

REQUIRED_PACKAGES = ["numpy", "pandas", "torch", "IPython"]
MISSING_PACKAGES = [
    package for package in REQUIRED_PACKAGES if importlib.util.find_spec(package) is None
]
if MISSING_PACKAGES:
    raise RuntimeError(
        "Notebook 03 requires these packages in the active kernel: "
        + ", ".join(MISSING_PACKAGES)
        + ". Run it in the project ML/Kaggle environment used for model notebooks."
    )

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import Markdown, display
from torch import nn


def show_records(records: Iterable[dict[str, Any]]) -> None:
    display(pd.DataFrame(list(records)))


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_inside(root: Path, relative_path: str) -> Path:
    resolved_root = root.resolve()
    resolved = (resolved_root / relative_path).resolve()
    resolved.relative_to(resolved_root)
    return resolved


def project_root(start: Path) -> Path:
    override = os.environ.get("COLDSTART_PROJECT_ROOT")
    if override:
        return Path(override).expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate.resolve()
    return start.resolve()


EXECUTION_CONTEXT = "kaggle" if Path("/kaggle/input").exists() else "local"
PROJECT_ROOT = project_root(Path.cwd())
WORKSPACE_ROOT = Path(
    os.environ.get(
        "COLDSTART_WORKSPACE_ROOT",
        "/kaggle/working" if EXECUTION_CONTEXT == "kaggle" else PROJECT_ROOT / ".notebook",
    )
).expanduser().resolve()
INPUT_ROOT = Path(
    os.environ.get(
        "COLDSTART_INPUT_ROOT",
        "/kaggle/input" if EXECUTION_CONTEXT == "kaggle" else PROJECT_ROOT / "data",
    )
).expanduser().resolve()
ARTIFACT_ROOT = Path(
    os.environ.get("COLDSTART_ARTIFACT_ROOT", WORKSPACE_ROOT / "artifacts")
).expanduser().resolve()
PROTOCOL_RELATIVE_MANIFEST = Path("protocols/ml-1m/coldstart-v1/manifest.json")
MODEL_OUTPUT_ROOT = ARTIFACT_ROOT / "models" / "ml-1m" / "lightgcn-v1"
FAST_DEV_RUN = os.environ.get("COLDSTART_FAST_DEV_RUN", "0") == "1"

requested_device = os.environ.get("COLDSTART_DEVICE")
if requested_device:
    DEVICE = torch.device(requested_device)
    if DEVICE.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("COLDSTART_DEVICE requests CUDA, but torch.cuda is unavailable")
else:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base_epochs = int(os.environ.get("COLDSTART_LIGHTGCN_EPOCHS", "12"))
base_sample_size = int(os.environ.get("COLDSTART_LIGHTGCN_SAMPLE_SIZE", "65536"))
if FAST_DEV_RUN:
    base_epochs = min(base_epochs, 2)
    base_sample_size = min(base_sample_size, 8192)

TARGET_SEEDS: tuple[int, ...] = (2025, 7788, 9999, 3407, 4517)
PHASES = ("Cold", "Warm A", "Warm B", "Warm C")
SCORING_BATCH_SIZE = int(os.environ.get("COLDSTART_LIGHTGCN_SCORE_BATCH", "131072"))
MAIN_LIGHTGCN_VARIANT = "LightGCN-implicit-all"


def make_run_config(seed: int, variant: str) -> dict[str, Any]:
    if variant == "LightGCN-implicit-all":
        edge_label_policy = "all_interactions"
        support_edge_policy = "all_interactions"
    elif variant == "LightGCN-positive-only":
        edge_label_policy = "positive_only"
        support_edge_policy = "positive_only"
    else:
        raise ValueError(f"Unknown variant: {variant}")

    return {
        "schema_version": "lightgcn-baseline-v1",
        "variant": variant,
        "edge_label_policy": edge_label_policy,
        "support_edge_policy": support_edge_policy,
        "seed": int(seed),
        "fast_dev_run": FAST_DEV_RUN,
        "candidate_configs": [
            {
                "embedding_dim": 64,
                "n_layers": 2,
                "learning_rate": 0.003,
                "weight_decay": 1e-4,
                "epochs": base_epochs,
                "sample_size": base_sample_size,
            },
            {
                "embedding_dim": 64,
                "n_layers": 3,
                "learning_rate": 0.003,
                "weight_decay": 1e-4,
                "epochs": base_epochs,
                "sample_size": base_sample_size,
            },
        ][:1 if FAST_DEV_RUN else 2],
        "phase_order": list(PHASES),
        "scoring_batch_size": SCORING_BATCH_SIZE,
        "training_feedback": f"BPR loss on edges defined by {edge_label_policy}",
        "negative_item_pool": "items visible in the current training base only",
        "selection_labels": "validation query labels only",
        "evaluation_thresholds": "frozen per-phase validation thresholds",
    }


RUN_CONFIGS = {seed: make_run_config(seed, MAIN_LIGHTGCN_VARIANT) for seed in TARGET_SEEDS}
RUN_CONFIG_HASHES = {
    seed: hashlib.sha256(
        json.dumps(
            config, sort_keys=True, separators=(",", ":"), allow_nan=False
        ).encode()
    ).hexdigest()
    for seed, config in RUN_CONFIGS.items()
}

show_records(
    [
        {
            "execution_context": EXECUTION_CONTEXT,
            "python": platform.python_version(),
            "torch": torch.__version__,
            "device": str(DEVICE),
            "artifact_root": str(ARTIFACT_ROOT),
            "model_output_root": str(MODEL_OUTPUT_ROOT),
            "target_seeds": list(TARGET_SEEDS),
            "run_config_sha256_by_seed": RUN_CONFIG_HASHES,
        }
    ]
)


def protocol_candidates() -> list[tuple[Path, Path]]:
    candidates: list[tuple[Path, Path]] = []
    explicit = os.environ.get("COLDSTART_PROTOCOL_ROOT")
    roots = [Path(explicit).expanduser()] if explicit else []
    roots.extend([PROJECT_ROOT / ".notebook" / "artifacts", ARTIFACT_ROOT])

    for root in roots:
        pointer = root / PROTOCOL_RELATIVE_MANIFEST
        if pointer.is_file():
            candidates.append((root.resolve(), pointer.resolve()))
        direct = root / "manifest.json"
        if root.name == "coldstart-v1" and direct.is_file():
            candidates.append((root.parents[2].resolve(), direct.resolve()))

    if INPUT_ROOT.is_dir():
        for pointer in sorted(INPUT_ROOT.rglob("manifest.json")):
            parent = pointer.parent
            if (
                parent.name == "coldstart-v1"
                and parent.parent.name == "ml-1m"
                and parent.parent.parent.name == "protocols"
            ):
                candidates.append((pointer.parents[3].resolve(), pointer.resolve()))

    unique: list[tuple[Path, Path]] = []
    seen: set[str] = set()
    for root, pointer in candidates:
        key = str(pointer)
        if key not in seen:
            seen.add(key)
            unique.append((root, pointer))
    return unique


def load_verified_protocol(root: Path, pointer: Path) -> tuple[dict[str, Any], dict[str, pd.DataFrame]]:
    pointer_bytes = pointer.read_bytes()
    manifest = json.loads(pointer_bytes)
    if manifest.get("protocol_schema_version") != "ml1m-coldstart-v1":
        raise ValueError(f"Unexpected protocol schema: {manifest.get('protocol_schema_version')!r}")
    if manifest.get("protocol_status") != "PASS":
        raise ValueError(f"Protocol status is not PASS: {manifest.get('protocol_status')!r}")
    checks = manifest.get("checks")
    if (
        not isinstance(checks, list)
        or not checks
        or not all(isinstance(row, dict) and row.get("status") == "PASS" for row in checks)
    ):
        raise ValueError("One or more notebook-02 protocol checks did not pass")

    bundle_id = manifest.get("bundle_id")
    if (
        not isinstance(bundle_id, str)
        or not bundle_id
        or bundle_id in {".", ".."}
        or Path(bundle_id).name != bundle_id
    ):
        raise ValueError(f"Invalid protocol bundle id: {bundle_id!r}")
    bundle_manifest = resolve_inside(root, manifest["bundle_manifest"])
    expected_bundle_manifest = (
        pointer.parent / "generations" / bundle_id / "manifest.json"
    ).resolve()
    if bundle_manifest != expected_bundle_manifest:
        raise ValueError("Protocol bundle_manifest is not the immutable generation manifest")
    if bundle_manifest.read_bytes() != pointer_bytes:
        raise ValueError("Protocol pointer and immutable generation manifest differ")

    tables: dict[str, pd.DataFrame] = {}
    for name, artifact in manifest["artifacts"].items():
        path = resolve_inside(root, artifact["path"])
        if not path.is_file() or sha256_file(path) != artifact["sha256"]:
            raise ValueError(f"Protocol artifact verification failed: {name}")
        schema = manifest["output_schemas"][name]
        table = pd.read_csv(path, dtype=schema["read_csv_dtypes"])
        if list(table.columns) != schema["columns"] or len(table) != artifact["rows"]:
            raise ValueError(f"Protocol table contract failed: {name}")
        tables[name] = table
    return manifest, tables


PROTOCOL_ERRORS: list[str] = []
PROTOCOL_ROOT = None
PROTOCOL_POINTER = None
PROTOCOL_MANIFEST = None
TABLES = None
for candidate_root, candidate_pointer in protocol_candidates():
    try:
        PROTOCOL_MANIFEST, TABLES = load_verified_protocol(candidate_root, candidate_pointer)
        PROTOCOL_ROOT, PROTOCOL_POINTER = candidate_root, candidate_pointer
        break
    except Exception as error:
        PROTOCOL_ERRORS.append(f"{candidate_pointer}: {error}")

if PROTOCOL_MANIFEST is None or TABLES is None or PROTOCOL_POINTER is None:
    raise RuntimeError(
        "No valid notebook-02 protocol bundle found. Set COLDSTART_PROTOCOL_ROOT. "
        + " | ".join(PROTOCOL_ERRORS)
    )

TUNING_TRAIN = TABLES["tuning_train"]
FINAL_TRAIN = TABLES["final_train"]
VALIDATION_TASKS = TABLES["validation_tasks"]
EVALUATION_TASKS = TABLES["evaluation_tasks"]
USERS = TABLES["users"]
ITEMS = TABLES["items"]
N_USERS = int(USERS["user_idx"].max()) + 1
N_ITEMS = int(ITEMS["item_idx"].max()) + 1
PHASE_SUPPORT_ROLES = {
    "Cold": [],
    "Warm A": ["warm_a"],
    "Warm B": ["warm_a", "warm_b"],
    "Warm C": ["warm_a", "warm_b", "warm_c"],
}
PROTOCOL_POINTER_SHA256 = sha256_file(PROTOCOL_POINTER)

show_records(
    [
        {
            "protocol_bundle": PROTOCOL_MANIFEST["bundle_id"],
            "protocol_pointer_sha256": PROTOCOL_POINTER_SHA256,
            "users": N_USERS,
            "items": N_ITEMS,
            "tuning_train_rows": len(TUNING_TRAIN),
            "final_train_rows": len(FINAL_TRAIN),
            "validation_query_rows": int(VALIDATION_TASKS["role"].eq("query").sum()),
            "evaluation_query_rows": int(EVALUATION_TASKS["role"].eq("query").sum()),
        }
    ]
)

,execution_context,python,torch,device,artifact_root,model_output_root,target_seeds,run_config_sha256_by_seed
0,local,3.12.3,2.11.0+cu128,cuda,/workspace/HungPH/coldstart-recsys/.notebook/a...,/workspace/HungPH/coldstart-recsys/.notebook/a...,"[2025, 7788, 9999, 3407, 4517]",{2025: 'ea9a20a69393952d1a7dc9a314a1b4ab5487f0...


,protocol_bundle,protocol_pointer_sha256,users,items,tuning_train_rows,final_train_rows,validation_query_rows,evaluation_query_rows
0,20260716T183148-e7b92de81a1e,5256c3aeda7f7e619fd60d9530397af6a207c7a83d80c4...,6040,2375,684728,854530,152762,57569


## LightGCN, Phase Graphs, and Metrics

LightGCN keeps only ID embeddings and normalized user-item propagation. The
base graph is `tuning_train` during selection and `final_train` after freezing
choices. Warm phases add support rows cumulatively, while Cold leaves new items
isolated instead of dropping them. F1 thresholds are fitted only on validation
query rows; evaluation uses those frozen thresholds unchanged.

In [2]:
EDGE_COLUMNS = ["user_idx", "item_idx"]


def edge_array(table: pd.DataFrame) -> np.ndarray:
    return table[EDGE_COLUMNS].to_numpy(dtype=np.int64, copy=True)


def phase_edges(base: pd.DataFrame, tasks: pd.DataFrame, phase: str, policy: str) -> np.ndarray:
    roles = PHASE_SUPPORT_ROLES[phase]
    if not roles:
        return edge_array(base)
    
    support = tasks.loc[tasks["role"].isin(roles)].copy()
    if policy == "positive_only":
        support = support[support["label"] == 1]
        
    return pd.concat([base[EDGE_COLUMNS], support[EDGE_COLUMNS]], ignore_index=True).to_numpy(
        dtype=np.int64, copy=True
    )


def build_norm_adj(edges: np.ndarray) -> torch.Tensor:
    users = torch.as_tensor(edges[:, 0], dtype=torch.long, device=DEVICE)
    items = torch.as_tensor(edges[:, 1] + N_USERS, dtype=torch.long, device=DEVICE)
    row = torch.cat([users, items])
    col = torch.cat([items, users])
    degree = torch.bincount(row, minlength=N_USERS + N_ITEMS).float()
    degree = degree.clamp_min_(1.0)
    values = torch.rsqrt(degree[row] * degree[col])
    indices = torch.stack([row, col])
    return torch.sparse_coo_tensor(
        indices, values, (N_USERS + N_ITEMS, N_USERS + N_ITEMS), device=DEVICE
    ).coalesce()


class LightGCN(nn.Module):
    def __init__(self, n_users: int, n_items: int, embedding_dim: int, n_layers: int):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.n_layers = n_layers
        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        self.item_embedding = nn.Embedding(n_items, embedding_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

    def propagate(self, norm_adj: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        embeddings = torch.cat([self.user_embedding.weight, self.item_embedding.weight], dim=0)
        layers = [embeddings]
        for _ in range(self.n_layers):
            embeddings = torch.sparse.mm(norm_adj, embeddings)
            layers.append(embeddings)
        output = torch.stack(layers, dim=0).mean(dim=0)
        return output[: self.n_users], output[self.n_users :]


def make_seen_sets(edges: np.ndarray) -> list[set[int]]:
    seen = [set() for _ in range(N_USERS)]
    for user_idx, item_idx in edges:
        seen[int(user_idx)].add(int(item_idx))
    return seen


def sample_negative_items(
    users: np.ndarray,
    item_pool: np.ndarray,
    seen_by_user: list[set[int]],
    rng: np.random.Generator,
) -> np.ndarray:
    negatives = rng.choice(item_pool, size=len(users), replace=True).astype(np.int64)
    bad = np.zeros(len(users), dtype=bool)
    for _ in range(20):
        bad = np.fromiter(
            (int(item) in seen_by_user[int(user)] for user, item in zip(users, negatives)),
            dtype=bool,
            count=len(users),
        )
        if not bad.any():
            return negatives
        negatives[bad] = rng.choice(item_pool, size=int(bad.sum()), replace=True)

    for idx in np.flatnonzero(bad):
        seen = seen_by_user[int(users[idx])]
        available = np.array([item for item in item_pool if int(item) not in seen], dtype=np.int64)
        negatives[idx] = rng.choice(available if len(available) else item_pool)
    return negatives


def train_lightgcn(
    train_df: pd.DataFrame,
    config: dict[str, Any],
    run_config: dict[str, Any],
    seed: int,
    run_name: str,
) -> tuple[LightGCN, pd.DataFrame, float]:
    start = time.perf_counter()
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    if run_config["edge_label_policy"] == "positive_only":
        train_edges = edge_array(train_df[train_df["label"] == 1])
    else: # "all_interactions"
        train_edges = edge_array(train_df)

    item_pool = np.unique(train_edges[:, 1]).astype(np.int64)
    seen_by_user = make_seen_sets(train_edges)
    norm_adj = build_norm_adj(train_edges)
    model = LightGCN(
        N_USERS,
        N_ITEMS,
        embedding_dim=int(config["embedding_dim"]),
        n_layers=int(config["n_layers"]),
    ).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=float(config["learning_rate"]))
    history: list[dict[str, Any]] = []
    sample_size = int(config["sample_size"])

    for epoch in range(1, int(config["epochs"]) + 1):
        model.train()
        sampled = rng.integers(0, len(train_edges), size=sample_size)
        users_np = train_edges[sampled, 0]
        positives_np = train_edges[sampled, 1]
        negatives_np = sample_negative_items(users_np, item_pool, seen_by_user, rng)

        users = torch.as_tensor(users_np, dtype=torch.long, device=DEVICE)
        positives = torch.as_tensor(positives_np, dtype=torch.long, device=DEVICE)
        negatives = torch.as_tensor(negatives_np, dtype=torch.long, device=DEVICE)

        optimizer.zero_grad(set_to_none=True)
        user_embeddings, item_embeddings = model.propagate(norm_adj)
        pos_scores = (user_embeddings[users] * item_embeddings[positives]).sum(dim=1)
        neg_scores = (user_embeddings[users] * item_embeddings[negatives]).sum(dim=1)
        bpr_loss = -F.logsigmoid(pos_scores - neg_scores).mean()
        regularizer = (
            model.user_embedding(users).pow(2).sum()
            + model.item_embedding(positives).pow(2).sum()
            + model.item_embedding(negatives).pow(2).sum()
        ) / (2.0 * len(users_np))
        loss = bpr_loss + float(config["weight_decay"]) * regularizer
        loss.backward()
        optimizer.step()

        history.append(
            {
                "run_name": run_name,
                "epoch": epoch,
                "loss": float(loss.detach().cpu()),
                "bpr_loss": float(bpr_loss.detach().cpu()),
                "regularizer": float(regularizer.detach().cpu()),
                "sample_size": sample_size,
            }
        )

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return model, pd.DataFrame(history), time.perf_counter() - start


def score_query_rows(
    model: LightGCN,
    graph_edges: np.ndarray,
    query_rows: pd.DataFrame,
    phase: str,
    scoring_batch_size: int,
) -> pd.DataFrame:
    model.eval()
    norm_adj = build_norm_adj(graph_edges)
    scores: list[np.ndarray] = []
    batch_size = int(scoring_batch_size)
    with torch.no_grad():
        user_embeddings, item_embeddings = model.propagate(norm_adj)
        users_np = query_rows["user_idx"].to_numpy(dtype=np.int64)
        items_np = query_rows["item_idx"].to_numpy(dtype=np.int64)
        for start in range(0, len(query_rows), batch_size):
            end = min(start + batch_size, len(query_rows))
            users = torch.as_tensor(users_np[start:end], dtype=torch.long, device=DEVICE)
            items = torch.as_tensor(items_np[start:end], dtype=torch.long, device=DEVICE)
            batch_scores = (user_embeddings[users] * item_embeddings[items]).sum(dim=1)
            scores.append(batch_scores.detach().cpu().numpy())
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    result = query_rows[
        ["source_row", "user_id", "user_idx", "item_id", "item_idx", "label"]
    ].copy()
    result.insert(0, "phase", phase)
    result["score"] = np.concatenate(scores).astype(np.float32)
    return result


def roc_auc(labels: np.ndarray, scores: np.ndarray) -> float:
    labels = labels.astype(np.int64)
    positives = int(labels.sum())
    negatives = int(len(labels) - positives)
    if positives == 0 or negatives == 0:
        return float("nan")

    order = np.argsort(scores, kind="mergesort")
    sorted_scores = scores[order]
    ranks = np.arange(1, len(scores) + 1, dtype=np.float64)
    start = 0
    while start < len(scores):
        end = start + 1
        while end < len(scores) and sorted_scores[end] == sorted_scores[start]:
            end += 1
        if end - start > 1:
            ranks[start:end] = ranks[start:end].mean()
        start = end
    original_ranks = np.empty_like(ranks)
    original_ranks[order] = ranks
    return float((original_ranks[labels == 1].sum() - positives * (positives + 1) / 2) / (positives * negatives))


def best_f1_threshold(labels: np.ndarray, scores: np.ndarray) -> tuple[float, float]:
    labels = labels.astype(np.int64)
    order = np.argsort(-scores, kind="mergesort")
    sorted_labels = labels[order]
    sorted_scores = scores[order]
    tp = np.cumsum(sorted_labels)
    fp = np.cumsum(1 - sorted_labels)
    fn = int(sorted_labels.sum()) - tp
    denominator = 2 * tp + fp + fn
    f1 = np.divide(2 * tp, denominator, out=np.zeros_like(tp, dtype=np.float64), where=denominator > 0)
    group_ends = np.flatnonzero(
        np.r_[sorted_scores[1:] != sorted_scores[:-1], True]
    )
    best = int(group_ends[np.argmax(f1[group_ends])])
    return float(sorted_scores[best]), float(f1[best])


def binary_metrics(labels: np.ndarray, scores: np.ndarray, threshold: float) -> dict[str, Any]:
    labels = labels.astype(np.int64)
    predictions = scores >= threshold
    
    # Core metrics
    tp = int(((predictions == 1) & (labels == 1)).sum())
    fp = int(((predictions == 1) & (labels == 0)).sum())
    fn = int(((predictions == 0) & (labels == 1)).sum())
    tn = int(((predictions == 0) & (labels == 0)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    model_f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    
    # Diagnostic metrics
    positive_rate = labels.mean()
    all_positive_predictions = np.ones_like(labels)
    all_pos_tp = int(((all_positive_predictions == 1) & (labels == 1)).sum())
    all_pos_fp = int(((all_positive_predictions == 1) & (labels == 0)).sum())
    all_positive_f1 = 2 * all_pos_tp / (2 * all_pos_tp + all_pos_fp) if (2*all_pos_tp + all_pos_fp) > 0 else 0.0
    
    roc_auc_score = roc_auc(labels, scores)
    roc_auc_negated_score = roc_auc(labels, -scores)

    scores_pos = scores[labels == 1]
    scores_neg = scores[labels == 0]

    return {
        "rows": int(len(labels)),
        "positives": int(labels.sum()),
        "threshold": float(threshold),
        "accuracy": float((tp + tn) / len(labels)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(model_f1), # For backward compatibility
        "roc_auc": roc_auc_score, # For backward compatibility
        "model_f1": float(model_f1),
        "roc_auc_score": roc_auc_score,
        "predicted_positive_rate": float(predictions.mean()),
        "score_pos_mean": float(np.mean(scores_pos)) if len(scores_pos) > 0 else 0.0,
        "score_neg_mean": float(np.mean(scores_neg)) if len(scores_neg) > 0 else 0.0,
        "score_pos_q05": float(np.quantile(scores_pos, 0.05)) if len(scores_pos) > 0 else 0.0,
        "score_pos_q50": float(np.quantile(scores_pos, 0.50)) if len(scores_pos) > 0 else 0.0,
        "score_pos_q95": float(np.quantile(scores_pos, 0.95)) if len(scores_pos) > 0 else 0.0,
        "score_neg_q05": float(np.quantile(scores_neg, 0.05)) if len(scores_neg) > 0 else 0.0,
        "score_neg_q50": float(np.quantile(scores_neg, 0.50)) if len(scores_neg) > 0 else 0.0,
        "score_neg_q95": float(np.quantile(scores_neg, 0.95)) if len(scores_neg) > 0 else 0.0,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        # Additional diagnostics
        "positive_rate": float(positive_rate),
        "all_positive_f1": float(all_positive_f1),
        "model_f1_minus_all_positive_f1": float(model_f1 - all_positive_f1),
        "roc_auc_negated_score": roc_auc_negated_score,
    }

def run_diagnosis(metrics: dict[str, Any]) -> str:
    if (abs(metrics["model_f1"] - metrics["all_positive_f1"]) < 0.005 and
        metrics["predicted_positive_rate"] > 0.98 and
        0.45 <= metrics["roc_auc_score"] <= 0.55):
        return "uninformative_all_positive_threshold"
    
    if (metrics["roc_auc_negated_score"] > metrics["roc_auc_score"] + 0.10 and
        metrics["roc_auc_negated_score"] > 0.60):
        return "possible_score_direction_issue"
        
    return "ok"



def summarize_predictions(
    predictions: pd.DataFrame,
    split: str,
    thresholds: dict[str, float] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    threshold_rows: list[dict[str, Any]] = []
    metric_rows: list[dict[str, Any]] = []
    for phase in PHASES:
        phase_predictions = predictions[predictions["phase"].eq(phase)]
        labels = phase_predictions["label"].to_numpy(dtype=np.int64)
        scores = phase_predictions["score"].to_numpy(dtype=np.float64)
        if thresholds is None:
            threshold, best_f1 = best_f1_threshold(labels, scores)
        else:
            threshold, best_f1 = float(thresholds[phase]), np.nan
        threshold_rows.append(
            {"split": split, "phase": phase, "threshold": threshold, "validation_best_f1": best_f1}
        )
        metric_rows.append(
            {"split": split, "phase": phase, **binary_metrics(labels, scores, threshold)}
        )
    return pd.DataFrame(threshold_rows), pd.DataFrame(metric_rows)


def numeric_tables_are_finite(*tables: pd.DataFrame) -> bool:
    for table in tables:
        numeric = table.select_dtypes(include=[np.number])
        if numeric.empty or not np.isfinite(numeric.to_numpy(dtype=np.float64)).all():
            return False
    return True

## Tune on Validation Only

For each seed, every candidate trains on `tuning_train`. Validation phase
graphs add only the matching old-validation support rows, and thresholds come
only from validation query labels. The chosen configuration is the one with
the best mean phase F1; evaluation labels are not accepted by this function.

In [3]:
TUNING_EDGES = edge_array(TUNING_TRAIN)
VALIDATION_QUERY = VALIDATION_TASKS[VALIDATION_TASKS["role"].eq("query")].reset_index(drop=True)


def select_on_validation(seed: int, run_config: dict[str, Any]) -> dict[str, Any]:
    training_history_parts: list[pd.DataFrame] = []
    validation_metric_parts: list[pd.DataFrame] = []
    validation_threshold_parts: list[pd.DataFrame] = []
    config_summary_rows: list[dict[str, Any]] = []
    selected_payload: dict[str, Any] | None = None

    for config_index, config in enumerate(run_config["candidate_configs"], start=1):
        config_name = f"cfg{config_index:02d}_L{config['n_layers']}_D{config['embedding_dim']}"
        model, history, training_seconds = train_lightgcn(
            TUNING_TRAIN, config, run_config, seed=seed + config_index, run_name=config_name
        )
        history["config_name"] = config_name
        training_history_parts.append(history)

        validation_prediction_parts = [
            score_query_rows(
                model,
                phase_edges(TUNING_TRAIN, VALIDATION_TASKS, phase, run_config["support_edge_policy"]),
                VALIDATION_QUERY,
                phase,
                int(run_config["scoring_batch_size"]),
            )
            for phase in PHASES
        ]
        validation_predictions = pd.concat(validation_prediction_parts, ignore_index=True)
        thresholds, metrics = summarize_predictions(validation_predictions, split="validation")
        thresholds["config_name"] = config_name
        metrics["config_name"] = config_name
        validation_threshold_parts.append(thresholds)
        validation_metric_parts.append(metrics)

        mean_f1 = float(metrics["f1"].mean())
        mean_auc = float(metrics["roc_auc"].mean())
        summary = {
            "config_name": config_name,
            "mean_validation_f1": mean_f1,
            "mean_validation_auc": mean_auc,
            "training_seconds": float(training_seconds),
            **config,
        }
        config_summary_rows.append(summary)
        candidate_payload = {
            "config_name": config_name,
            "config": dict(config),
            "summary": summary,
            "thresholds": thresholds,
            "metrics": metrics,
            "predictions": validation_predictions,
        }
        if selected_payload is None or (mean_f1, mean_auc) > (
            selected_payload["summary"]["mean_validation_f1"],
            selected_payload["summary"]["mean_validation_auc"],
        ):
            selected_payload = candidate_payload

        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    if selected_payload is None:
        raise RuntimeError(f"No LightGCN candidate was trained for seed {seed}")

    return {
        "training_history": pd.concat(training_history_parts, ignore_index=True),
        "validation_metrics_all": pd.concat(validation_metric_parts, ignore_index=True),
        "validation_thresholds_all": pd.concat(validation_threshold_parts, ignore_index=True),
        "config_summary": pd.DataFrame(config_summary_rows).sort_values(
            ["mean_validation_f1", "mean_validation_auc"], ascending=False
        ),
        "selected_config_name": str(selected_payload["config_name"]),
        "selected_config": dict(selected_payload["config"]),
        "selected_summary": dict(selected_payload["summary"]),
        "validation_predictions": selected_payload["predictions"].copy(),
        "validation_thresholds": selected_payload["thresholds"].copy(),
        "validation_metrics": selected_payload["metrics"].copy(),
        "threshold_by_phase": dict(
            zip(selected_payload["thresholds"]["phase"], selected_payload["thresholds"]["threshold"])
        ),
    }

## Final Refit, New-Item Evaluation, and Export

Sequentially complete selection, refit, evaluation, publication, and
verification for each exact target seed. Every evaluation uses that seed's
frozen validation thresholds. Quality warnings are recorded per phase without
weakening protocol, schema, hash, or completeness failures.

In [4]:
FINAL_EDGES = edge_array(FINAL_TRAIN)
EVALUATION_QUERY = EVALUATION_TASKS[EVALUATION_TASKS["role"].eq("query")].reset_index(drop=True)
def json_ready(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_ready(item) for item in value]
    if hasattr(value, "item"):
        return value.item()
    return str(value)


def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temporary.write_text(content, encoding="utf-8")
    temporary.replace(path)


def write_csv(path: Path, table: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    table.to_csv(temporary, index=False, lineterminator="\n")
    temporary.replace(path)


def relative_output(path: Path) -> str:
    return str(path.resolve().relative_to(ARTIFACT_ROOT.resolve()))


def csv_schema(table: pd.DataFrame) -> dict[str, Any]:
    def dtype_name(dtype: Any) -> str:
        name = str(dtype)
        return "string" if name in {"str", "string"} or name.startswith("string") else name

    return {
        "columns": list(table.columns),
        "read_csv_dtypes": {column: dtype_name(dtype) for column, dtype in table.dtypes.items()},
    }


def evaluation_quality_checks(evaluation_metrics: pd.DataFrame) -> list[dict[str, Any]]:
    checks: list[dict[str, Any]] = []
    for phase in PHASES:
        phase_row = evaluation_metrics.loc[evaluation_metrics["phase"].eq(phase)].iloc[0]
        positive_rate = float(phase_row["predicted_positive_rate"])
        phase_auc = float(phase_row["roc_auc"])
        checks.extend(
            [
                {
                    "check": "predicted positive rate is below suspicious saturation",
                    "split": "evaluation",
                    "phase": phase,
                    "metric": "predicted_positive_rate",
                    "status": "WARN" if positive_rate >= 0.98 else "PASS",
                    "observed": positive_rate,
                    "expected": "< 0.98",
                    "severity": "WARN",
                },
                {
                    "check": "ROC-AUC is outside near-random interval",
                    "split": "evaluation",
                    "phase": phase,
                    "metric": "roc_auc",
                    "status": "WARN" if 0.45 <= phase_auc <= 0.55 else "PASS",
                    "observed": phase_auc,
                    "expected": "outside inclusive interval [0.45, 0.55]",
                    "severity": "WARN",
                },
            ]
        )
    return checks


def verify_seed_generation(
    seed: int,
    manifest: dict[str, Any],
    manifest_text: str,
    output_tables: dict[str, pd.DataFrame],
) -> list[dict[str, Any]]:
    checks: list[dict[str, Any]] = []

    def verify(name: str, condition: bool, observed: Any, expected: Any) -> None:
        checks.append(
            {
                "check": name,
                "status": "PASS" if condition else "FAIL",
                "observed": observed,
                "expected": expected,
            }
        )

    bundle_manifest_path = resolve_inside(ARTIFACT_ROOT, manifest["bundle_manifest"])
    manifest_matches = bundle_manifest_path.is_file() and bundle_manifest_path.read_text(
        encoding="utf-8"
    ) == manifest_text
    verify("immutable manifest matches", manifest_matches, manifest_matches, True)
    verify(
        "manifest run seed",
        int(manifest.get("run_config", {}).get("seed", -1)) == seed,
        manifest.get("run_config", {}).get("seed"),
        seed,
    )

    artifact_errors: list[str] = []
    for name, artifact in manifest["artifacts"].items():
        try:
            path = resolve_inside(ARTIFACT_ROOT, artifact["path"])
            if not path.is_file() or sha256_file(path) != artifact["sha256"]:
                artifact_errors.append(name)
        except Exception as error:
            artifact_errors.append(f"{name}: {error}")
    verify("artifact hashes verify", not artifact_errors, artifact_errors, [])

    table_errors: list[str] = []
    for name, expected_table in output_tables.items():
        try:
            artifact = manifest["artifacts"][name]
            schema = manifest["output_schemas"][name]
            table = pd.read_csv(
                resolve_inside(ARTIFACT_ROOT, artifact["path"]),
                dtype=schema["read_csv_dtypes"],
            )
            if list(table.columns) != schema["columns"] or len(table) != len(expected_table):
                table_errors.append(name)
        except Exception as error:
            table_errors.append(f"{name}: {error}")
    verify("CSV schemas and row counts verify", not table_errors, table_errors, [])

    selected_config_error = ""
    try:
        selected_path = resolve_inside(ARTIFACT_ROOT, manifest["artifacts"]["selected_config"]["path"])
        selected_payload = json.loads(selected_path.read_text(encoding="utf-8"))
        if selected_payload.get("run_config_sha256") != manifest["run_config_sha256"]:
            selected_config_error = "run_config_sha256 mismatch"
    except Exception as error:
        selected_config_error = str(error)
    verify("selected config contract verifies", not selected_config_error, selected_config_error, "")
    verify(
        "checkpoint exported",
        "final_model_checkpoint" in manifest["artifacts"],
        "final_model_checkpoint" in manifest["artifacts"],
        True,
    )
    return checks


def run_seed(seed: int) -> dict[str, Any]:
    if seed not in TARGET_SEEDS:
        raise ValueError(f"Unexpected LightGCN target seed: {seed}")
    run_config = make_run_config(seed, MAIN_LIGHTGCN_VARIANT)
    run_config_hash = hashlib.sha256(
        json.dumps(
            run_config, sort_keys=True, separators=(",", ":"), allow_nan=False
        ).encode()
    ).hexdigest()
    if run_config != RUN_CONFIGS[seed] or run_config_hash != RUN_CONFIG_HASHES[seed]:
        raise RuntimeError(f"Run configuration drift detected for seed {seed}")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    selection = select_on_validation(seed, run_config)
    selected_config_name = selection["selected_config_name"]
    selected_config = selection["selected_config"]
    selected_summary = selection["selected_summary"]
    threshold_by_phase = selection["threshold_by_phase"]

    final_model, final_history, final_training_seconds = train_lightgcn(
        FINAL_TRAIN,
        selected_config,
        run_config,
        seed=seed + 10_000,
        run_name=f"final_refit_{selected_config_name}",
    )
    final_history["config_name"] = selected_config_name
    training_history = pd.concat(
        [selection["training_history"], final_history], ignore_index=True
    )

    phase_graphs = {
        phase: phase_edges(FINAL_TRAIN, EVALUATION_TASKS, phase, run_config["support_edge_policy"]) for phase in PHASES
    }
    evaluation_predictions = pd.concat(
        [
            score_query_rows(
                final_model,
                phase_graphs[phase],
                EVALUATION_QUERY,
                phase,
                int(run_config["scoring_batch_size"]),
            )
            for phase in PHASES
        ],
        ignore_index=True,
    )
    _, evaluation_metrics = summarize_predictions(
        evaluation_predictions,
        split="evaluation",
        thresholds=threshold_by_phase,
    )
    evaluation_metrics["config_name"] = selected_config_name

    diagnostics = pd.DataFrame(
        [
            {
                "phase": phase,
                "base_train_edges": len(FINAL_TRAIN),
                "support_edges": int(
                    EVALUATION_TASKS["role"].isin(PHASE_SUPPORT_ROLES[phase]).sum()
                ),
                "graph_edges": len(phase_graphs[phase]),
                "query_rows": int(EVALUATION_QUERY.shape[0]),
                "query_users": int(EVALUATION_QUERY["user_idx"].nunique()),
                "query_items": int(EVALUATION_QUERY["item_idx"].nunique()),
                "threshold": float(threshold_by_phase[phase]),
            }
            for phase in PHASES
        ]
    )
    diagnostics["final_training_seconds"] = float(final_training_seconds)
    diagnostics["device"] = str(DEVICE)

    pre_export_checks: list[dict[str, Any]] = []

    def pre_export_check(name: str, condition: bool, observed: Any, expected: Any) -> None:
        pre_export_checks.append(
            {
                "check": name,
                "status": "PASS" if condition else "FAIL",
                "observed": observed,
                "expected": expected,
            }
        )

    expected_validation_predictions = int(VALIDATION_QUERY.shape[0] * len(PHASES))
    expected_evaluation_predictions = int(EVALUATION_QUERY.shape[0] * len(PHASES))
    final_excludes_evaluation_items = set(FINAL_TRAIN["item_idx"]).isdisjoint(
        set(EVALUATION_TASKS["item_idx"])
    )
    evaluation_thresholds = dict(
        zip(evaluation_metrics["phase"], evaluation_metrics["threshold"])
    )
    expected_graph_rows = {
        phase: len(FINAL_TRAIN)
        + int(EVALUATION_TASKS["role"].isin(PHASE_SUPPORT_ROLES[phase]).sum())
        for phase in PHASES
    }
    observed_graph_rows = {phase: len(phase_graphs[phase]) for phase in PHASES}

    pre_export_check(
        "protocol schema",
        PROTOCOL_MANIFEST["protocol_schema_version"] == "ml1m-coldstart-v1",
        PROTOCOL_MANIFEST["protocol_schema_version"],
        "ml1m-coldstart-v1",
    )
    pre_export_check("run seed is target seed", run_config["seed"] == seed, run_config["seed"], seed)
    pre_export_check(
        "phase thresholds complete",
        set(threshold_by_phase) == set(PHASES),
        sorted(threshold_by_phase),
        sorted(PHASES),
    )
    pre_export_check(
        "evaluation thresholds remain frozen",
        evaluation_thresholds == threshold_by_phase,
        evaluation_thresholds,
        threshold_by_phase,
    )
    pre_export_check(
        "validation predictions complete",
        len(selection["validation_predictions"]) == expected_validation_predictions,
        len(selection["validation_predictions"]),
        expected_validation_predictions,
    )
    pre_export_check(
        "evaluation predictions complete",
        len(evaluation_predictions) == expected_evaluation_predictions,
        len(evaluation_predictions),
        expected_evaluation_predictions,
    )
    pre_export_check(
        "evaluation phases complete",
        set(evaluation_metrics["phase"]) == set(PHASES) and len(evaluation_metrics) == len(PHASES),
        sorted(evaluation_metrics["phase"]),
        sorted(PHASES),
    )
    pre_export_check(
        "evaluation metrics finite",
        bool(
            np.isfinite(
                evaluation_metrics[["threshold", "f1", "roc_auc", "predicted_positive_rate"]]
                .to_numpy(dtype=np.float64)
            ).all()
        ),
        "all finite",
        "all finite",
    )
    pre_export_check(
        "final train excludes evaluation items",
        final_excludes_evaluation_items,
        final_excludes_evaluation_items,
        True,
    )
    pre_export_check(
        "Cold graph keeps evaluation items isolated",
        np.array_equal(phase_graphs["Cold"], FINAL_EDGES),
        len(phase_graphs["Cold"]),
        len(FINAL_EDGES),
    )
    pre_export_check(
        "Warm graphs add cumulative support only",
        observed_graph_rows == expected_graph_rows,
        observed_graph_rows,
        expected_graph_rows,
    )
    training_values_finite = numeric_tables_are_finite(training_history)
    validation_values_finite = numeric_tables_are_finite(
        selection["config_summary"],
        selection["validation_thresholds"],
        selection["validation_metrics"],
        selection["validation_predictions"],
    )
    evaluation_values_finite = numeric_tables_are_finite(
        evaluation_predictions, evaluation_metrics
    )
    pre_export_check(
        "training values finite",
        training_values_finite,
        training_values_finite,
        True,
    )
    pre_export_check(
        "validation values finite",
        validation_values_finite,
        validation_values_finite,
        True,
    )
    pre_export_check(
        "evaluation values finite",
        evaluation_values_finite,
        evaluation_values_finite,
        True,
    )

    if not all(row["status"] == "PASS" for row in pre_export_checks):
        display(pd.DataFrame(pre_export_checks))
        raise RuntimeError(f"LightGCN semantic checks failed for seed {seed}; artifacts not published")

    quality_checks = evaluation_quality_checks(evaluation_metrics)
    quality_warning_count = sum(row["status"] == "WARN" for row in quality_checks)
    quality_status = "WARN" if quality_warning_count else "PASS"

    output_tables = {
        "training_history": training_history,
        "config_summary": selection["config_summary"],
        "validation_thresholds": selection["validation_thresholds"],
        "validation_metrics": selection["validation_metrics"],
        "validation_predictions": selection["validation_predictions"],
        "evaluation_metrics": evaluation_metrics,
        "evaluation_predictions": evaluation_predictions,
        "diagnostics": diagnostics,
    }
    selected_config_payload = {
        "schema_version": run_config["schema_version"],
        "selected_config_name": selected_config_name,
        "selected_config": selected_config,
        "selected_summary": selected_summary,
        "threshold_by_phase": threshold_by_phase,
        "run_config_sha256": run_config_hash,
        "protocol_pointer_sha256": PROTOCOL_POINTER_SHA256,
    }

    bundle_id = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
        + f"-s{seed}-"
        + uuid.uuid4().hex[:12]
    )
    staging_root = MODEL_OUTPUT_ROOT / f".staging-{bundle_id}"
    generation_root = MODEL_OUTPUT_ROOT / "generations" / bundle_id
    staging_root.mkdir(parents=True, exist_ok=False)
    output_artifacts: dict[str, Any] = {}
    generation_published = False

    try:
        for name, table in output_tables.items():
            staging_path = staging_root / f"{name}.csv"
            published_path = generation_root / f"{name}.csv"
            write_csv(staging_path, table)
            output_artifacts[name] = {
                "path": relative_output(published_path),
                "sha256": sha256_file(staging_path),
                "rows": len(table),
            }

        selected_config_path = staging_root / "selected_config.json"
        write_text(
            selected_config_path,
            json.dumps(
                json_ready(selected_config_payload),
                indent=2,
                sort_keys=True,
                allow_nan=False,
            )
            + "\n",
        )
        output_artifacts["selected_config"] = {
            "path": relative_output(generation_root / "selected_config.json"),
            "sha256": sha256_file(selected_config_path),
        }

        checkpoint_path = staging_root / "final_model.pt"
        torch.save(
            {
                "schema_version": run_config["schema_version"],
                "seed": seed,
                "model_state_dict": final_model.state_dict(),
                "selected_config": selected_config,
                "threshold_by_phase": threshold_by_phase,
                "n_users": N_USERS,
                "n_items": N_ITEMS,
                "protocol_pointer_sha256": PROTOCOL_POINTER_SHA256,
            },
            checkpoint_path,
        )
        output_artifacts["final_model_checkpoint"] = {
            "path": relative_output(generation_root / "final_model.pt"),
            "sha256": sha256_file(checkpoint_path),
        }

        manifest = {
            "model_schema_version": run_config["schema_version"],
            "model_status": quality_status,
            "quality_status": quality_status,
            "quality_checks": quality_checks,
            "model_name": "LightGCN",
            "bundle_id": bundle_id,
            "bundle_manifest": relative_output(generation_root / "manifest.json"),
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "upstream_protocol": {
                "schema_version": PROTOCOL_MANIFEST["protocol_schema_version"],
                "bundle_id": PROTOCOL_MANIFEST["bundle_id"],
                "pointer_sha256": PROTOCOL_POINTER_SHA256,
            },
            "run_config": run_config,
            "run_config_sha256": run_config_hash,
            "selected_config": selected_config_payload,
            "summary": {
                "selected_config_name": selected_config_name,
                "mean_validation_f1": selected_summary["mean_validation_f1"],
                "mean_validation_auc": selected_summary["mean_validation_auc"],
                "mean_evaluation_f1": float(evaluation_metrics["f1"].mean()),
                "mean_evaluation_auc": float(evaluation_metrics["roc_auc"].mean()),
                "evaluation_prediction_rows": len(evaluation_predictions),
                "final_training_seconds": float(final_training_seconds),
                "quality_warning_count": quality_warning_count,
            },
            "artifacts": output_artifacts,
            "checks": pre_export_checks,
            "output_schemas": {name: csv_schema(table) for name, table in output_tables.items()},
            "phase_contract": PROTOCOL_MANIFEST["phase_contract"],
            "training_contract": {
                "tuning_base": "tuning_train from notebook 02",
                "final_refit_base": "final_train from notebook 02",
                "graph_support": "Cold none; Warm A/B/C cumulative support rows",
                "features": "ID embeddings only; no side information",
                "thresholds": "selected on validation query rows only and frozen for evaluation",
                "quality_diagnostics": "evaluation labels are diagnostic only; never used for selection",
            },
        }

        manifest_text = (
            json.dumps(
                json_ready(manifest), indent=2, sort_keys=True, allow_nan=False
            )
            + "\n"
        )
        write_text(staging_root / "manifest.json", manifest_text)
        generation_root.parent.mkdir(parents=True, exist_ok=True)
        staging_root.replace(generation_root)
        generation_published = True

        verification_checks = verify_seed_generation(
            seed, manifest, manifest_text, output_tables
        )
        if not all(row["status"] == "PASS" for row in verification_checks):
            display(pd.DataFrame(verification_checks))
            raise RuntimeError(
                f"Published LightGCN generation failed verification for seed {seed}"
            )
    except Exception:
        if staging_root.exists():
            shutil.rmtree(staging_root, ignore_errors=True)
        if generation_published and generation_root.exists():
            shutil.rmtree(generation_root)
        raise

    del final_model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return {
        "seed": seed,
        "bundle_id": bundle_id,
        "model_status": quality_status,
        "quality_warning_count": quality_warning_count,
        "selected_config_name": selected_config_name,
        "mean_evaluation_f1": float(evaluation_metrics["f1"].mean()),
        "mean_evaluation_auc": float(evaluation_metrics["roc_auc"].mean()),
        "artifact_count": len(output_artifacts),
        "generation_root": generation_root,
        "manifest": manifest,
        "manifest_text": manifest_text,
        "verification_checks": verification_checks,
        "verified": True,
    }


SEED_RUNS: list[dict[str, Any]] = []
for target_seed in TARGET_SEEDS:
    try:
        SEED_RUNS.append(run_seed(target_seed))
    except Exception as error:
        display(Markdown("### Notebook 03 LightGCN baseline: BLOCKED"))
        raise RuntimeError(f"LightGCN seed {target_seed} failed") from error

if tuple(run["seed"] for run in SEED_RUNS) != TARGET_SEEDS:
    display(Markdown("### Notebook 03 LightGCN baseline: BLOCKED"))
    raise RuntimeError("Exact target-seed execution was not completed")

LIGHTGCN_POINTER = MODEL_OUTPUT_ROOT / "manifest.json"
previous_pointer_text = (
    LIGHTGCN_POINTER.read_text(encoding="utf-8") if LIGHTGCN_POINTER.is_file() else None
)
try:
    write_text(LIGHTGCN_POINTER, SEED_RUNS[-1]["manifest_text"])
    if LIGHTGCN_POINTER.read_text(encoding="utf-8") != SEED_RUNS[-1]["manifest_text"]:
        raise RuntimeError("LightGCN pointer does not match the last verified generation")
except Exception:
    if previous_pointer_text is None:
        LIGHTGCN_POINTER.unlink(missing_ok=True)
    else:
        write_text(LIGHTGCN_POINTER, previous_pointer_text)
    raise
LIGHTGCN_MANIFEST = SEED_RUNS[-1]["manifest"]
SEED_REGISTRY = pd.DataFrame(
    [
        {
            "seed": run["seed"],
            "bundle_id": run["bundle_id"],
            "model_status": run["model_status"],
            "quality_warnings": run["quality_warning_count"],
            "selected_config": run["selected_config_name"],
            "mean_evaluation_f1": run["mean_evaluation_f1"],
            "mean_evaluation_auc": run["mean_evaluation_auc"],
            "verified": run["verified"],
        }
        for run in SEED_RUNS
    ]
)
display(SEED_REGISTRY)

/tmp/ipykernel_13175/1468539708.py:27: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_coo_tensor(
/tmp/ipykernel_13175/1468539708.py:168: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  users = torch.as_tensor(users_np[start:end], dtype=torch.long, devi

,seed,bundle_id,model_status,quality_warnings,selected_config,mean_evaluation_f1,mean_evaluation_auc,verified
0,2025,20260716T183339-s2025-a370f5a1659c,WARN,8,cfg01_L2_D64,0.579735,0.503708,True
1,7788,20260716T183349-s7788-055a93bf6705,WARN,8,cfg02_L3_D64,0.579768,0.486746,True
2,9999,20260716T183359-s9999-6cc7ff772669,WARN,8,cfg02_L3_D64,0.579773,0.477061,True
3,3407,20260716T183408-s3407-d3c5f4a86f19,WARN,8,cfg02_L3_D64,0.579759,0.492823,True
4,4517,20260716T183419-s4517-388725ed3236,WARN,8,cfg01_L2_D64,0.579735,0.498506,True


## Checks and Continuation

Before moving on, require the exact seed sequence and verify every immutable
generation plus the last-seed pointer. Quality warnings remain visible in each
source manifest for notebook 07, but only semantic/artifact failures block.

In [5]:
LIGHTGCN_CHECKS: list[dict[str, Any]] = []


def check(name: str, condition: bool, observed: Any, expected: Any) -> None:
    LIGHTGCN_CHECKS.append(
        {"check": name, "status": "PASS" if condition else "FAIL", "observed": observed, "expected": expected}
    )


observed_seeds = tuple(int(run["seed"]) for run in SEED_RUNS)
bundle_ids = [str(run["bundle_id"]) for run in SEED_RUNS]
manifest_seeds = tuple(int(run["manifest"]["run_config"]["seed"]) for run in SEED_RUNS)
generation_checks_pass = all(
    all(row["status"] == "PASS" for row in run["verification_checks"]) for run in SEED_RUNS
)
model_statuses_valid = all(run["model_status"] in {"PASS", "WARN"} for run in SEED_RUNS)
quality_statuses_consistent = all(
    run["manifest"]["model_status"] == run["manifest"]["quality_status"]
    and run["manifest"]["model_status"]
    == ("WARN" if any(row["status"] == "WARN" for row in run["manifest"]["quality_checks"]) else "PASS")
    for run in SEED_RUNS
)
quality_phase_coverage = all(
    {
        (row["phase"], row["metric"])
        for row in run["manifest"]["quality_checks"]
    }
    == {
        (phase, metric)
        for phase in PHASES
        for metric in ("predicted_positive_rate", "roc_auc")
    }
    for run in SEED_RUNS
)
pointer_matches_last_seed = LIGHTGCN_POINTER.read_text(encoding="utf-8") == SEED_RUNS[-1][
    "manifest_text"
]

check("exact target seeds exported in order", observed_seeds == TARGET_SEEDS, observed_seeds, TARGET_SEEDS)
check("one unique generation per seed", len(set(bundle_ids)) == len(TARGET_SEEDS), bundle_ids, len(TARGET_SEEDS))
check("manifest seeds match targets", manifest_seeds == TARGET_SEEDS, manifest_seeds, TARGET_SEEDS)
check("all generations verify", generation_checks_pass, generation_checks_pass, True)
check("model statuses are PASS or WARN", model_statuses_valid, [run["model_status"] for run in SEED_RUNS], ["PASS", "WARN"])
check("quality statuses are consistent", quality_statuses_consistent, quality_statuses_consistent, True)
check("quality checks cover every phase", quality_phase_coverage, quality_phase_coverage, True)
check("manifest pointer matches last seed", pointer_matches_last_seed, pointer_matches_last_seed, True)

LIGHTGCN_EXPORTS_VALID = all(row["status"] == "PASS" for row in LIGHTGCN_CHECKS)
ANY_QUALITY_WARNING = any(run["model_status"] == "WARN" for run in SEED_RUNS)
NOTEBOOK_STATUS = (
    "BLOCKED" if not LIGHTGCN_EXPORTS_VALID else "WARN" if ANY_QUALITY_WARNING else "READY"
)
display(pd.DataFrame(LIGHTGCN_CHECKS))
display(Markdown("### Notebook 03 LightGCN baseline: " + NOTEBOOK_STATUS))

if NOTEBOOK_STATUS == "BLOCKED":
    raise RuntimeError("LightGCN multi-seed exports are missing or invalid; inspect LIGHTGCN_CHECKS")

display(
    Markdown(
        "**Next notebook:** implement EmerG in notebook 04 with the same protocol manifest. "
        "Notebook 07 can read each LightGCN manifest's PASS/WARN `model_status` and quality evidence."
    )
)

,check,status,observed,expected
0,exact target seeds exported in order,PASS,"(2025, 7788, 9999, 3407, 4517)","(2025, 7788, 9999, 3407, 4517)"
1,one unique generation per seed,PASS,"[20260716T183339-s2025-a370f5a1659c, 20260716T...",5
2,manifest seeds match targets,PASS,"(2025, 7788, 9999, 3407, 4517)","(2025, 7788, 9999, 3407, 4517)"
3,all generations verify,PASS,True,True
4,model statuses are PASS or WARN,PASS,"[WARN, WARN, WARN, WARN, WARN]","[PASS, WARN]"
5,quality statuses are consistent,PASS,True,True
6,quality checks cover every phase,PASS,True,True
7,manifest pointer matches last seed,PASS,True,True


### Notebook 03 LightGCN baseline: WARN

**Next notebook:** implement EmerG in notebook 04 with the same protocol manifest. Notebook 07 can read each LightGCN manifest's PASS/WARN `model_status` and quality evidence.